# Tools with Claude

---

## The Problem Without Tools

By default, Claude only knows information from its **training data**.
It cannot access current events, real-time data, or external systems.
```
User:  "What's the weather in San Francisco?"
Claude: "I'm sorry, I don't have access to up-to-date weather information."
```

Tools solve this by giving Claude a structured way to **request and receive fresh data**.

---

## How Tool Use Works — The Flow
```
1. You send Claude a question + tool instructions
         |
         v
2. Claude decides it needs external data
   → sends a structured Tool Request
         |
         v
3. Your server runs code to fetch the data
   (calls a weather API, queries a database, etc.)
         |
         v
4. You send the retrieved data back to Claude
         |
         v
5. Claude generates a final response using
   the original question + fresh data
```

All of these steps happen **within a single user interaction**.

---

## Weather Example — Step by Step

| Step | What happens |
|---|---|
| User asks | "What's the weather in San Francisco?" |
| You send | Question + instructions on how to call a weather tool |
| Claude requests | Weather data for San Francisco specifically |
| Your server | Calls a real weather API, gets live conditions |
| You return | The weather data to Claude |
| Claude responds | A complete, accurate, current weather answer |

---

## Key Benefits

| Benefit | Detail |
|---|---|
| **Real-time information** | Access data that didn't exist during training |
| **External system integration** | Connect to APIs, databases, services |
| **Dynamic responses** | Answers based on the latest available data |
| **Structured interaction** | Claude knows exactly what to ask for and how |

---

## What Tools Enable

Without tools → Claude is a **static knowledge base**  
With tools → Claude is a **dynamic assistant** that works with live data

Use cases: weather, stock prices, database queries, search, calendar data,
internal APIs, and any other real-time information your users need.

---

## Key Takeaway

> Tool use follows a structured **request → retrieve → respond** loop.
> Claude does not call tools itself — it asks your server to do so,
> then uses the result to generate its final answer.
> This keeps Claude in control of reasoning while your code handles data fetching.

# Project — Building a Reminder System with Tools

---

## Goal

Accept natural language like:
```
"Set a reminder for my doctor's appointment. It's a week from Thursday."
```

And have Claude respond:
```
"OK, I will remind you on [exact date]."
```

---

## Why This Is Challenging

Claude has three specific gaps that prevent this from working out of the box:

| Limitation | Problem |
|---|---|
| **Limited time awareness** | Claude may know the date but not the exact current time |
| **Date calculation issues** | Claude is unreliable at time-based arithmetic (e.g. "3 weeks from next Friday") |
| **No reminder capability** | Claude has no built-in mechanism to actually set a reminder |

Each gap needs its own tool.

---

## The Three Tools

| Tool | Purpose |
|---|---|
| `get_current_datetime` | Give Claude the precise current date and time |
| `add_duration_to_datetime` | Reliably calculate a future date from a duration |
| `set_reminder` | Actually store/trigger the reminder in the system |

---

## How They Work Together
```
User: "Remind me in a week from Thursday"
         |
         v
Claude calls get_current_datetime  → knows today is Wed March 26
         |
         v
Claude calls add_duration_to_datetime("next Thursday + 7 days")
         → returns exact target datetime
         |
         v
Claude calls set_reminder(datetime, message)
         → reminder is stored
         |
         v
Claude responds: "OK, I will remind you on April 3rd."
```

---

## Design Principle

> When Claude has a limitation, **extend its capabilities through tools**
> rather than trying to work around the limitation in the prompt.

Tools are how you bridge the gap between what Claude knows natively
and what your application needs it to do.

---

## Implementation Approach

Build and test **one tool at a time**:
```
Step 1 → get_current_datetime   (simplest — no inputs needed)
Step 2 → add_duration_to_datetime
Step 3 → set_reminder
Step 4 → combine all three in a single prompt
```

This incremental approach lets you verify each tool works in isolation
before combining them into the full reminder workflow.

# Tool Functions — Implementation

---

## What is a Tool Function?

A plain Python function that gets **automatically executed** when Claude decides it
needs extra data to help a user.
```
User asks: "What time is it?"
     |
     v
Claude calls → get_current_datetime()
     |
     v
Your function runs → returns "2024-01-15 14:30:25"
     |
     v
Claude uses the result to answer the user
```

---

## Best Practices

| Rule | Why it matters |
|---|---|
| Use descriptive function and parameter names | Claude reads these to understand what the tool does |
| Validate all inputs | Claude can learn from errors and retry with corrected values |
| Provide meaningful error messages | `"Location cannot be empty"` is more useful than a generic crash |

> Claude **sees your error messages** — a clear error can cause it to retry the
> call with corrected parameters automatically.

---

## Tool Function 1 — `get_current_datetime`
```python
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)
```

### Usage
```python
# Default format
get_current_datetime()
# → "2024-01-15 14:30:25"

# Time only
get_current_datetime("%H:%M")
# → "14:30"

# Date only
get_current_datetime("%Y-%m-%d")
# → "2024-01-15"
```

---

## What Comes Next

Creating the function is only **step 1** of integrating a tool with Claude.
```
Step 1 → Write the Python function         ← done above
Step 2 → Write a JSON schema describing the function to Claude
Step 3 → Register the tool in your API call
Step 4 → Handle Claude's tool call request in your code
Step 5 → Return the result back to Claude
```

The JSON schema is how Claude knows:
- What the tool is called
- What parameters it accepts
- What each parameter means
- Which parameters are required

---

## Key Takeaway

> Tool functions are just regular Python functions — the complexity is in
> **describing them to Claude** via JSON schema and **wiring them into**
> the message loop so Claude can call them and receive results.

# Tool Step 2 — Writing a JSON Schema

---

## The 5-Step Tool Integration Pipeline
```
Write a tool function
        |
        v
Write a JSON Schema    ← current step
        |
        v
Call Claude with JSON Schema
        |
        v
Run Tool
        |
        v
Add Tool Result and call Claude again
```

---

## What is a JSON Schema?

A standard data validation specification (not AI-specific) that describes what
arguments a function accepts. Claude reads this to understand **when** and **how**
to call your tool.

---

## Schema Structure — Three Parts
```json
{
    "name": "get_weather",
    "description": "Retrieves current weather...",
    "input_schema": {
        "type": "object",
        "properties": {
            "location": {
                "type": "string",
                "description": "The location for which..."
            }
        },
        "required": ["location"]
    }
}
```

| Field | Purpose |
|---|---|
| `name` | Identifies the tool — Claude uses this to call it |
| `description` | Tells Claude what the tool does and **when to use it** |
| `input_schema` | Describes the function's arguments and their types |

---

## Writing a Good Description

The description is what Claude reads to decide whether to use the tool.

- Aim for **3–4 sentences**
- Explain **what the tool does**
- Explain **when Claude should use it**
- Explain **what it returns**
- Add detailed descriptions for **each parameter**

---

## The Easy Way — Let Claude Write the Schema

Instead of writing it by hand:

1. Copy your tool function code
2. Ask Claude: *"Write a valid JSON schema spec for tool calling for this function.
   Follow best practices from the Anthropic tool use documentation."*
3. Paste the generated schema into your code

---

## Full Example — `get_current_datetime`
```python
from datetime import datetime

def get_current_datetime(date_format="%Y-%m-%d %H:%M:%S"):
    if not date_format:
        raise ValueError("date_format cannot be empty")
    return datetime.now().strftime(date_format)

get_current_datetime_schema = {
    "name": "get_current_datetime",
    "description": (
        "Returns the current date and time formatted according to the specified format. "
        "Use this tool when the user asks about the current time or date, or when you "
        "need to know the current datetime to perform calculations. "
        "Returns a formatted datetime string."
    ),
    "input_schema": {
        "type": "object",
        "properties": {
            "date_format": {
                "type": "string",
                "description": (
                    "A string specifying the format of the returned datetime. "
                    "Uses Python's strftime format codes. "
                    "Example: '%Y-%m-%d %H:%M:%S' for full datetime."
                ),
                "default": "%Y-%m-%d %H:%M:%S"
            }
        },
        "required": []    # date_format is optional — has a default
    }
}
```

**Naming convention:** always pair `function_name` with `function_name_schema`.

---

## Adding Type Safety (Optional but Recommended)
```python
from anthropic.types import ToolParam

get_current_datetime_schema = ToolParam({
    "name": "get_current_datetime",
    "description": "...",
    "input_schema": { ... }
})
```

- Prevents type errors when passing the schema to the API
- Makes your code more robust — catches schema mistakes early
- Not required for functionality, but good practice

---

## Key Takeaways

- The schema is **documentation for Claude** — quality of description directly
  affects whether Claude knows when to use the tool
- Use Claude to **generate your schemas** from function code — faster and more accurate
- Always follow `function_name` / `function_name_schema` naming
- Use `ToolParam` for type safety when working with the Anthropic SDK

# Tool Step 3 — Call Claude with JSON Schema

---

## What Changes in the API Call

Add a `tools` parameter — a list of JSON schemas describing available functions:
```python
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema],   # <-- new parameter
)
```

---

## Multi-Block Messages

When Claude decides to use a tool, the response is **no longer a single text block**.
The assistant message now contains a **list of blocks** (also called "parts").

| Block Type | Content |
|---|---|
| **Text block** | Human-readable explanation — "Let me find that for you..." |
| **ToolUse block** | Instructions for your code: which tool to call + what parameters |

### ToolUse block structure:
```python
ToolUseBlock(
    id='toolu_01QtKMDNkU1FBj73NP2FbF8w',   # unique ID for this tool call
    input={'date_format': '%H:%M:%S'},        # parameters to pass to the function
    name='get_current_datetime',              # function to call
    type='tool_use'
)
```

---

## Preserving Multi-Block History

Claude has no memory — you must manually append **the full content list**
(not just the text) to maintain conversation context:
```python
# Correct — preserves both text block AND tool use block
messages.append({
    "role": "assistant",
    "content": response.content    # full content list, not just response.content[0].text
})
```

If you only save the text block, Claude loses track of which tool it called,
and subsequent calls will break.

---

## The Complete Tool Usage Flow (All 5 Steps)
```
1. Send user message + tool schema to Claude
         |
         v
2. Receive assistant message
   → Text block: "Let me find that for you..."
   → ToolUse block: {name, input, id}
         |
         v
3. Extract tool name + input from ToolUse block
   → Execute the actual Python function
         |
         v
4. Send tool result back to Claude
   + full conversation history (including assistant's multi-block message)
         |
         v
5. Receive Claude's final response with the real data
```

---

## Updating Helper Functions

Your existing `add_assistant_message()` likely only handles plain text.
It needs to support multi-block content:
```python
def add_assistant_message(messages, content):
    # content can now be a string OR a list of blocks
    messages.append({
        "role": "assistant",
        "content": content    # pass response.content directly
    })
```

---

## Key Takeaways

- Add `tools=[...]` to `client.messages.create()` to enable tool use
- Claude's response will now contain **multiple blocks** — text + tool use
- Always append `response.content` (the full list) to message history — never just the text
- The `ToolUse` block tells your code **exactly** which function to call and with what args

# Tool Steps 4 & 5 — Run Tool + Return Result to Claude

---

## Step 4 — Run the Tool

Extract the input parameters from Claude's `ToolUse` block and call the function:
```python
# Claude's tool use block is at response.content[1]
tool_input = response.content[1].input      # dict of arguments
tool_name  = response.content[1].name       # "get_current_datetime"
tool_id    = response.content[1].id         # unique ID — needed for the result block

# Call the function using dict unpacking
result = get_current_datetime(**tool_input)
```

`**tool_input` unpacks the dictionary into keyword arguments so the function
receives them correctly.

---

## Step 5 — Send the Result Back (Tool Result Block)

The result goes inside a **user message** as a `tool_result` block:
```python
messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_id,       # must match the ToolUse block's id
        "content": str(result),       # output serialized as a string
        "is_error": False             # True if the function raised an error
    }]
})
```

### Tool Result Block Fields

| Field | Purpose |
|---|---|
| `tool_use_id` | Links this result to the specific `ToolUse` block that requested it |
| `content` | The function's output, serialized as a string |
| `is_error` | `True` if an error occurred during execution |

---

## Handling Multiple Tool Calls

Claude can request **multiple tools in a single response** (e.g. two separate calculations).
Each has a unique `id`. You must send back one result block per tool call, matched by ID:
```python
content_blocks = []

for block in response.content:
    if block.type == "tool_use":
        result = get_current_datetime(**block.input)
        content_blocks.append({
            "type": "tool_result",
            "tool_use_id": block.id,      # matches each specific tool call
            "content": str(result),
            "is_error": False
        })

messages.append({"role": "user", "content": content_blocks})
```

---

## Making the Final API Call

Always include the tool schema even on the follow-up call — Claude needs it to
understand the tool references already in the conversation history:
```python
final_response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_schema]    # still required
)

print(final_response.content[0].text)
# → "The current time is 15:04:22."
```

---

## Complete Message History at This Point
```
messages = [
    {"role": "user",      "content": "What is the exact time, formatted as HH:MM:SS?"},
    {"role": "assistant", "content": [TextBlock(...), ToolUseBlock(...)]},
    {"role": "user",      "content": [{"type": "tool_result", "tool_use_id": "...", ...}]}
]
```

---

## Key Takeaways

- Use `**block.input` to unpack tool arguments into your Python function
- The `tool_use_id` in the result block **must match** the ID in the `ToolUse` block
- Serialize all tool output to a **string** before putting it in `content`
- Always pass the tool schema on the follow-up call
- For multiple tool calls, send back **one result block per tool call** in a single user message

# Multi-Turn Conversations with Tools

---

## The Problem

Some user questions require Claude to call **multiple tools in sequence**:
```
User: "What day is 103 days from today?"

1. Claude calls get_current_datetime  → gets today's date
2. Claude calls add_duration_to_datetime → calculates target date
3. Claude gives final answer
```

Your code needs a loop that keeps running until Claude stops requesting tools.

---

## The Conversation Loop Pattern
```python
def run_conversation(messages, tools):
    while True:
        response = chat(messages, tools=tools)
        add_assistant_message(messages, response)

        # If Claude isn't asking for a tool, we're done
        if response.stop_reason != "tool_use":
            break

        # Otherwise, run the requested tools and add results
        tool_result_blocks = run_tools(response)
        add_user_message(messages, tool_result_blocks)

    return messages
```

The loop continues as long as `stop_reason == "tool_use"`.
When Claude has enough information it stops requesting tools and the loop exits.

---

## Refactoring 1 — Update `add_user_message` and `add_assistant_message`

Helper functions must now handle strings, block lists, **and** full Message objects:
```python
from anthropic.types import Message

def add_user_message(messages, message):
    messages.append({
        "role": "user",
        "content": message.content if isinstance(message, Message) else message
    })

def add_assistant_message(messages, message):
    messages.append({
        "role": "assistant",
        "content": message.content if isinstance(message, Message) else message
    })
```

---

## Refactoring 2 — Update `chat()` to Accept Tools and Return Full Message
```python
def chat(messages, system=None, temperature=1.0, stop_sequences=[], tools=None):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
        "stop_sequences": stop_sequences,
    }

    if tools:
        params["tools"] = tools

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message                   # return full message, not just .content[0].text
```

---

## Refactoring 3 — Add `text_from_message()` Helper

Since `chat()` now returns a full message object, extract readable text when needed:
```python
def text_from_message(message):
    return "\n".join(
        [block.text for block in message.content if block.type == "text"]
    )
```

Joins all text blocks from a message into a single readable string.

---

## Summary of Changes

| Function | Before | After |
|---|---|---|
| `add_user_message` | Accepts string only | Accepts string, block list, or Message object |
| `add_assistant_message` | Accepts string only | Accepts string, block list, or Message object |
| `chat()` | Returns `content[0].text` | Returns full `Message` object; accepts `tools=` |
| `text_from_message()` | Did not exist | Extracts all text blocks from a Message |

---

## Key Takeaway

> Multi-tool conversations require a **while loop** that keeps calling Claude
> and running tools until `stop_reason != "tool_use"`.  
> The helper function refactors above make this loop clean and reusable
> across any number of tools.

# Implementing Multi-Turn Tool Conversations

---

## The Core Loop
```python
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[get_current_datetime_schema])
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break                          # Claude has a final answer — exit loop

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages
```

---

## `stop_reason` — The Exit Signal

| Value | Meaning |
|---|---|
| `"tool_use"` | Claude needs to call a tool — keep looping |
| `"end_turn"` | Claude has finished its assistant message — exit loop |
| `"max_tokens"` | Hit token output limit — exit loop |
| `"stop_sequence"` | Encountered a stop sequence — exit loop |

The loop only continues when `stop_reason == "tool_use"`.

---

## Multiple ToolUse Blocks in One Response

Claude can request **multiple tools in a single response**.
`message.content` is a list that may contain one TextBlock and multiple ToolUseBlocks:
```
message.content = [
    TextBlock(text="I'll solve these", type='text'),
    ToolUseBlock(id='ab3', input={'expression': '10 + 10'}, name='calculator', type='tool_use'),
    ToolUseBlock(id='po9', input={'expression': '30 + 30'}, name='calculator', type='tool_use'),
]
```

Process **every** ToolUseBlock — not just the first one.

---

## `run_tools()` — Processing All Tool Requests
```python
def run_tools(message):
    tool_requests = [
        block for block in message.content if block.type == "tool_use"
    ]
    tool_result_blocks = []

    for tool_request in tool_requests:
        try:
            tool_output = run_tool(tool_request.name, tool_request.input)
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,    # must match ToolUseBlock id
                "content": json.dumps(tool_output),
                "is_error": False
            })
        except Exception as e:
            tool_result_blocks.append({
                "type": "tool_result",
                "tool_use_id": tool_request.id,
                "content": f"Error: {e}",
                "is_error": True
            })

    return tool_result_blocks
```

---

## `run_tool()` — Routing to the Right Function

Maps tool names to their Python implementations:
```python
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")
```

Add new tools here — the core loop never needs to change.

---

## ToolUseBlock vs ToolResultBlock — Quick Reference

| Field | ToolUseBlock (Claude → You) | ToolResultBlock (You → Claude) |
|---|---|---|
| `id` / `tool_use_id` | ID of this tool call | Must match the ToolUseBlock ID |
| `input` / `content` | Dict of arguments | Output string from the function |
| `name` | Tool function to call | — |
| `is_error` | — | `True` if function raised an error |
| `type` | `"tool_use"` | `"tool_result"` |

---

## Complete Flow (End to End)
```
1. Add user message → "What day is 103 days from today?"
         |
         v
2. Call Claude with tools → response (stop_reason = "tool_use")
         |
         v
3. run_tools(response) → executes get_current_datetime
         |
         v
4. Add tool results as user message → call Claude again
         |
         v
5. Claude calls add_duration_to_datetime → stop_reason = "tool_use" again
         |
         v
6. run_tools(response) → executes add_duration_to_datetime
         |
         v
7. Add tool results → call Claude again
         |
         v
8. Claude responds with final answer → stop_reason = "end_turn"
         |
         v
9. Loop exits → return messages
```

---

## Key Takeaways

- Use `stop_reason == "tool_use"` as the loop condition
- Always loop over **all** blocks in `message.content` — there may be multiple ToolUseBlocks
- Match every `ToolUseBlock.id` to a `ToolResultBlock.tool_use_id`
- Always send a result block even on error — set `is_error: True`
- Keep tool routing in a separate `run_tool()` function — makes adding new tools easy

# Adding Multiple Tools to the Reminder System

---

## The Three Tools

| Tool | Purpose |
|---|---|
| `get_current_datetime` | Give Claude the precise current date and time |
| `add_duration_to_datetime` | Reliably calculate a future date from a duration |
| `set_reminder` | Store the reminder in the system |

---

## Step 1 — Register All Tools in `run_conversation`
```python
def run_conversation(messages):
    while True:
        response = chat(messages, tools=[
            get_current_datetime_schema,
            add_duration_to_datetime_schema,
            set_reminder_schema
        ])
        add_assistant_message(messages, response)
        print(text_from_message(response))

        if response.stop_reason != "tool_use":
            break

        tool_results = run_tools(response)
        add_user_message(messages, tool_results)

    return messages
```

---

## Step 2 — Add All Tools to the Router
```python
def run_tool(tool_name, tool_input):
    if tool_name == "get_current_datetime":
        return get_current_datetime(**tool_input)
    elif tool_name == "add_duration_to_datetime":
        return add_duration_to_datetime(**tool_input)
    elif tool_name == "set_reminder":
        return set_reminder(**tool_input)
    else:
        raise ValueError(f"Unknown tool: {tool_name}")
```

Each new tool = one `elif` block. The rest of the pipeline never changes.

---

## Testing — Multi-Tool Request

**Input:**
```
"Set a reminder for my doctor's appointment. It's 177 days after Jan 1st, 2050."
```

**What Claude does:**
```
1. Calls add_duration_to_datetime("Jan 1 2050", 177 days)
         → returns "June 27, 2050"

2. Calls set_reminder("June 27, 2050", "Doctor's appointment")
         → reminder stored

3. Final response: "I've set a reminder for your doctor's appointment
                    on June 27, 2050."
```

---

## Resulting Message History
```
messages = [
    {"role": "user",      "content": "Set a reminder for my doctor's..."},
    {"role": "assistant", "content": [TextBlock, ToolUseBlock(add_duration...)]},
    {"role": "user",      "content": [ToolResultBlock("June 27, 2050")]},
    {"role": "assistant", "content": [TextBlock, ToolUseBlock(set_reminder...)]},
    {"role": "user",      "content": [ToolResultBlock("Reminder set")]},
    {"role": "assistant", "content": [TextBlock("Done! Reminder set for...")]},
]
```

---

## The Pattern for Adding Any New Tool
```
1. Write the tool function
2. Write the JSON schema  (function_name_schema)
3. Add schema to tools=[] list in run_conversation
4. Add elif case in run_tool
```

Nothing else in the pipeline needs to change.

---

## Key Takeaway

> The modular design means adding new tools is purely additive —
> write the function, write the schema, register in two places.
> The conversation loop, error handling, and result formatting
> all work automatically for every new tool.

# Streaming with Tool Use & Fine-Grained Tool Calling

---

## Streaming + Tools — New Event Type

When streaming is enabled with tools, a new event type appears alongside the familiar
`ContentBlockDelta`:

| Event type | When it fires | Key properties |
|---|---|---|
| `text` / `ContentBlockDelta` | Regular text generation | `.text` |
| `input_json` / `InputJsonEvent` | Tool argument generation | `.partial_json`, `.snapshot` |
```python
for chunk in stream:
    if chunk.type == "input_json":
        print(chunk.partial_json)    # latest chunk of JSON
        current_args = chunk.snapshot  # cumulative JSON so far
```

---

## How JSON Validation Works (Default Behavior)

The Anthropic API does **not** send every chunk immediately. It buffers and validates
chunks before forwarding them.

**Rule:** Wait for a complete top-level key-value pair → validate → send all buffered chunks.

**Example — given this tool output structure:**
```json
{
  "abstract": "This paper presents...",
  "meta": { "word_count": 1455, "review": "Great paper" }
}
```
```
API receives chunks for "abstract" key
         |
         v
Full value complete? → Yes → validate → send all "abstract" chunks at once
         |
         v
API receives chunks for "meta" key
         |
         v
Full nested object complete? → Yes → validate → send all "meta" chunks at once
```

This causes visible **delays followed by bursts** — expected behaviour with default streaming.

---

## Fine-Grained Tool Calling

Disables API-side JSON validation — chunks are sent **as soon as Claude generates them**,
with no buffering.
```python
run_conversation(
    messages,
    tools=[save_article_schema],
    fine_grained=True              # <-- enable fine-grained
)
```

### Trade-offs

| | Default (validated) | Fine-grained |
|---|---|---|
| **Speed** | Delayed (waits for full keys) | Immediate |
| **JSON validity** | Guaranteed valid | Not guaranteed |
| **Error handling needed** | No | Yes — your code must handle bad JSON |

---

## Handling Invalid JSON (Required for Fine-Grained)

Claude may generate invalid JSON like `"word_count": undefined`.
Always wrap snapshot parsing in a try/except:
```python
try:
    parsed_args = json.loads(chunk.snapshot)
except json.JSONDecodeError:
    print("Partial/invalid JSON — skipping this chunk")
    # continue accumulating until valid
```

---

## When to Use Fine-Grained Tool Calling

Use it when:
- You need to show users **real-time progress** on tool argument generation
- You want to **start processing partial results** as fast as possible
- Buffering delays negatively impact UX

Stick with default when:
- JSON validity is critical to downstream processing
- You don't want to implement custom JSON error handling
- Responsiveness is not a priority

---

## Key Takeaway

> Default streaming with tools buffers until full key-value pairs are valid —
> reliable but adds latency.  
> Fine-grained tool calling removes that buffer for maximum speed,
> but shifts the responsibility for JSON validation entirely to your code.

# The Text Editor Tool

---

## What It Is

A tool **built directly into Claude** — the schema is pre-defined, but you must
provide the actual function implementations that execute the file operations.

| Custom tools | Text editor tool |
|---|---|
| You write the JSON schema | Claude already knows the schema |
| You write the function | You still write the function |

---

## What It Can Do

- View file or directory contents
- View a specific range of lines in a file
- Replace text in a file
- Create new files
- Insert text at specific lines in a file
- Undo recent edits to files

---

## Each Tool Still Has a Paired Schema + Function
```
add_duration_to_datetime_schema  ↔  def add_duration_to_datetime()
get_current_datetime_schema      ↔  def get_current_datetime()
set_reminder_schema              ↔  def set_reminder()
```

The text editor tool follows the same pattern — Claude provides the schema side,
you provide the implementation side.

---

## Schema Stub (Model-Dependent)

You pass a small stub to the API — Claude expands it into the full tool spec internally.
The exact stub depends on the model version:
```python
def get_text_edit_schema(model):
    if model.startswith("claude-3-7-sonnet"):
        return {
            "type": "text_editor_20250124",
            "name": "str_replace_editor",
        }
    elif model.startswith("claude-3-5-sonnet"):
        return {
            "type": "text_editor_20241022",
            "name": "str_replace_editor",
        }
```

> Full version strings for all models:
> https://docs.anthropic.com/en/docs/agents-and-tools/tool-use/text-editor-tool

---

## Example Usage

**Prompt:** "Open `./main.py` and summarize its contents"
```
1. Claude calls text editor tool → view ./main.py
2. Reads file contents
3. Returns a summary to the user
```

**Prompt:** "Add a function to calculate pi to 5 digits in `main.py`,
then create `test.py` to test it"
```
1. Claude views main.py
2. Replaces content with updated version including pi function
3. Creates test.py with unit tests
```

---

## When to Use It

- Building applications that need to **programmatically edit files**
- Working in environments **without a full code editor**
- Integrating file editing into **Claude-powered applications**
- Replicating AI code editor functionality with **fine-grained control**

---

## Key Takeaway

> The text editor tool gives Claude the ability to act like a software engineer
> working in a file system. The schema is built in — you only need to supply
> the implementations that actually perform each file operation on your machine.


## What is the Text Editor Tool?

Think of it like giving Claude **hands** to directly open, read, and edit files on your computer — instead of just *telling* you what to change, Claude can *actually do it*.

---

## The Core Idea: Schema Already Built In

Normally when you give Claude a custom tool, you have to define **both**:
1. The schema (what the tool looks like — inputs, parameters, etc.)
2. The function (what actually runs on your machine)

With the text editor tool, **Anthropic already wrote the schema** for you. You only need to write the function implementations (the code that actually reads/writes files on your system).

You "activate" it by passing a tiny stub to the API:

```python
{
    "type": "text_editor_20250728",
    "name": "str_replace_based_edit_tool"
}
```

That's it. Claude knows the rest internally.

---

## What Commands Can It Do?

| Command | What it does |
|---|---|
| `view` | Read a file or list a folder's contents |
| `str_replace` | Find exact text → replace it with new text |
| `create` | Create a brand new file with content |
| `insert` | Insert text at a specific line number |

*(Note: older versions had `undo_edit` too, but Claude 4 removed it)*

---

## How Does a Full Flow Work?

Say you tell Claude: *"Fix the syntax error in primes.py"*

```
You → "Fix the syntax error in primes.py"

Claude → calls view("primes.py")
         [your code reads the file, sends contents back]

Claude → spots missing colon on line 19
         calls str_replace(old="range(2, limit + 1)", new="range(2, limit + 1):")
         [your code makes the actual edit]

Claude → "Done! I fixed the missing colon on line 19."
```

Claude acts like a software engineer with a terminal — it reads, thinks, then edits precisely.

---

## The Key Distinction: Claude Decides, YOU Execute

This is the part most people miss. Claude never directly touches your filesystem. It **tells your code** what to do via tool calls, and **your Python/Node code** actually performs the file operation and returns the result. Claude is the brain; your script is the hands.

---

## Which Tool Version to Use?

| Model | Tool type string |
|---|---|
| Claude 4 (Opus, Sonnet) | `text_editor_20250728` |
| Claude Sonnet 3.7 | `text_editor_20250124` |
| Claude Sonnet 3.5 (old) | `text_editor_20241022` |

Each costs **700 extra input tokens** on top of normal usage.

---

## When Would YOU Use This? (Practical for your work)

Given your Plume migration work, this tool is very relevant:

- **Auto-fix transformed notebooks** — Claude reviews a `.ipynb` file, spots issues, fixes them inline
- **Batch patch transformation scripts** — Instead of manually editing `s3_to_github_direct_prod.py` after each bug, Claude could view → patch → verify in one loop
- **Code review + edit** — Claude reads a file, refactors it, writes the new version back

The key use case is anywhere you want Claude to be an **autonomous agent that edits files**, not just a chatbot that suggests edits.

# The Web Search Tool

---

## Important Note

> Your organization must enable the Web Search tool in the settings console before use.
> Settings: https://console.anthropic.com/settings/privacy

---

## What It Is

A **built-in tool** — unlike custom tools, Claude handles the entire search process
automatically. You only need to provide a small schema to enable it.
```
Our Server → Claude → Web (search query)
                  ← Search results
           ← Final answer with citations
```

---

## Schema Setup
```python
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5              # limits total searches per conversation
}
```

- `max_uses` prevents runaway API calls — Claude may do follow-up searches based on
  initial results, so cap it appropriately
- Add to the `tools=[]` list in your API call like any other tool schema

---

## Restricting to Specific Domains

Useful for authoritative or evidence-based sources:
```python
web_search_schema = {
    "type": "web_search_20250305",
    "name": "web_search",
    "max_uses": 5,
    "allowed_domains": ["nih.gov"]    # only search within these domains
}
```

Examples: `nih.gov` for medical research, `docs.python.org` for Python docs,
any official/trusted domain for your use case.

---

## Response Block Types

When Claude uses web search, the response contains multiple block types:

| Block type | What it contains |
|---|---|
| `TextBlock` | Claude's explanation of what it's doing |
| `ServerToolUseBlock` | The exact search query Claude used |
| `WebSearchToolResultBlock` | Container for all search results |
| `WebSearchResultBlock` | Individual results with titles and URLs |
| Citation blocks | Specific text Claude used to support its answer + source URL |

---

## Rendering the Response

| Block | How to render |
|---|---|
| Text blocks | Regular content / prose |
| Web search results | List of sources (shown at top or in sidebar) |
| Citations | Inline with text — show domain, page title, URL, quoted text |

Citations provide **transparency** — users can see exactly which source each
piece of information came from.

---

## Best Use Cases

- Current events and recent developments
- Specialized information not in Claude's training data
- Fact-checking against authoritative sources
- Research requiring up-to-date information

---

## Key Takeaway

> The web search tool requires **no implementation** — just the schema.
> Claude decides when to search, what to search for, and how many searches are needed.
> Use `max_uses` and `allowed_domains` to keep it focused and cost-controlled.